<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_03_Verified_Source_Evidence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, hashlib, json, math, os, platform, re, subprocess, sys
import pandas as pd

UTC_NOW = datetime.now(timezone.utc).isoformat()
IS_COLAB = Path('/content').exists()
RUN = Path('/content' if IS_COLAB else Path.cwd()) / 'ICHI2027_phase03'
PRIVATE = RUN / 'private_runtime_only'
PRIVATE.mkdir(parents=True, exist_ok=True)
DRIVE_ROOT = Path('/content/drive/MyDrive/NeuroFHIR_ICHI2027')
DRIVE_OK = False
if IS_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        DRIVE_OK = Path('/content/drive/MyDrive').is_dir()
    except Exception as exc:
        print('Drive unavailable:', type(exc).__name__)
DEST = DRIVE_ROOT / 'phase03_outputs' if DRIVE_OK else RUN / 'shareable_outputs'
DEST.mkdir(parents=True, exist_ok=True)
FILES = {
    'longitudinal': 'TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv',
    'repeat_pairs': 'TRACE_ADNI1_repeat_pairs_REAL.csv',
    'landmark12': 'TRACE_ADNI1_landmark12_REAL.csv',
}
print(json.dumps({'phase':'03_verified_source_evidence','drive_output_enabled':DRIVE_OK,
                  'shareable_output_folder':str(DEST),'utc':UTC_NOW}, indent=2))

Mounted at /content/drive
{
  "phase": "03_verified_source_evidence",
  "drive_output_enabled": true,
  "shareable_output_folder": "/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase03_outputs",
  "utc": "2026-09-26T22:27:31.841236+00:00"
}


In [2]:
def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1048576), b''): h.update(chunk)
    return h.hexdigest()

def present(s):
    return ~s.fillna('').astype(str).str.strip().str.lower().isin(
        {'','na','n/a','nan','null','none','unknown','unavailable','not available'})

def safe_count(value):
    n = int(value)
    return '<5' if 0 < n < 5 else n

def clean(s): return s.fillna('').astype(str).str.strip()

def count_missing(df, cols):
    return {c:{'present':safe_count(present(df[c]).sum()),'missing':safe_count((~present(df[c])).sum())}
            for c in cols if c in df.columns}

PHASE02 = None
if DRIVE_OK:
    audit = DRIVE_ROOT / 'phase02_outputs' / 'ICHI2027_Phase02_SHAREABLE.json'
    if audit.is_file():
        PHASE02 = json.loads(audit.read_text())
        if PHASE02.get('phase') != '02_real_source_reconciliation':
            raise ValueError('Existing Phase02 audit does not match project phase.')
print('Previous Phase02 output discovered:', PHASE02 is not None)

Previous Phase02 output discovered: True


In [3]:
search_folders = [Path('/content'),RUN/'private_runtime_only',
                  DRIVE_ROOT/'inputs',DRIVE_ROOT/'private_inputs',
                  Path('/content/ichi2027_neuroimaging_phase02/private_input'),
                  Path('/content/ichi2027_neuroimaging_phase01/authorized_input')]
locations={}
for role, name in FILES.items():
    matches=[p/name for p in search_folders if p.is_dir() and (p/name).is_file()]
    if matches and len({sha256_file(p) for p in matches})>1:
        raise RuntimeError('STOP: different contents found for '+name+'; remove duplicate versions before running.')
    if matches: locations[role]=matches[0]
if IS_COLAB:
    from google.colab import files as colab_files
    for role, name in FILES.items():
        if role in locations: continue
        print('Required authorized source file:',name)
        uploaded=colab_files.upload()
        if name not in uploaded:
            raise FileNotFoundError('STOP: missing required source file: '+name)
        path=PRIVATE/name
        path.write_bytes(uploaded[name])
        locations[role]=path
        del uploaded
if set(locations)!=set(FILES):
    raise FileNotFoundError('STOP: all three original source CSVs are required. Missing roles: '+repr(set(FILES)-set(locations)))
print('Three real source files found. No source rows are printed or copied into the shareable output.')

Three real source files found. No source rows are printed or copied into the shareable output.


In [4]:
REQUIRED={
 'longitudinal':{'RID','PTID','BL_VISCODE','M12_VISCODE','BL_EXAMDATE','M12_EXAMDATE',
                 'BL_IMAGEUID','M12_IMAGEUID','BL_LONIUID','M12_LONIUID',
                 'BL_LHIPPOC','BL_RHIPPOC','M12_LHIPPOC','M12_RHIPPOC'},
 'landmark12':{'RID','PTID','BL_EXAMDATE','M12_EXAMDATE','BL_IMAGEUID','M12_IMAGEUID',
               'BL_LHIPPOC','BL_RHIPPOC','M12_LHIPPOC','M12_RHIPPOC'},
 'repeat_pairs':{'RID','VISCODE','EXAMDATE','IMAGEUID_1','IMAGEUID_2','LONIUID_1','LONIUID_2'},
}
DATA={}
HASHES={}
for role,path in locations.items():
    with path.open(encoding='utf-8-sig',newline='') as f:
        header=next(csv.reader(f))
    if len(header)!=len(set(header)):
        raise ValueError('STOP: duplicate column names in '+FILES[role])
    df=pd.read_csv(path,dtype=str,keep_default_na=False,low_memory=False,encoding='utf-8-sig')
    missing=REQUIRED[role]-set(df.columns)
    if missing:
        raise ValueError('STOP: required fields absent in '+FILES[role]+': '+repr(sorted(missing)))
    DATA[role]=df
    HASHES[role]={'sha256':sha256_file(path),'bytes':path.stat().st_size}
    print(role,{'source_rows':len(df),'source_columns':len(df.columns),
                'required_source_columns_present':True})
if PHASE02:
    for role,df in DATA.items():
        prior=PHASE02['input_profiles'][role]
        if len(df)!=prior['rows'] or len(df.columns)!=prior['columns']:
            raise ValueError('STOP: real file differs from the previously audited '+role+' input.')

longitudinal {'source_rows': 258, 'source_columns': 51, 'required_source_columns_present': True}
repeat_pairs {'source_rows': 90, 'source_columns': 51, 'required_source_columns_present': True}
landmark12 {'source_rows': 199, 'source_columns': 70, 'required_source_columns_present': True}


In [5]:
KEY_REPORT={}
IDS={}
for role,df in DATA.items():
    rid=clean(df['RID'])
    valid=present(df['RID'])
    IDs=rid[valid]
    IDS[role]=set(IDs)
    item={
      'rows':int(len(df)),
      'valid_rid_rows':safe_count(valid.sum()),
      'missing_rid_rows':safe_count((~valid).sum()),
      'distinct_rid':safe_count(IDs.nunique()),
      'duplicate_rid_rows':safe_count(IDs.duplicated(keep=False).sum()),
      'rid_whitespace_rows':safe_count((df['RID']!=rid).sum()),
    }
    if 'PTID' in df:
        pairs=pd.DataFrame({'rid':rid,'ptid':clean(df['PTID'])})
        pairs=pairs[valid & present(df['PTID'])]
        item['rid_with_multiple_ptid']=safe_count((pairs.groupby('rid').ptid.nunique()>1).sum())
        item['ptid_with_multiple_rid']=safe_count((pairs.groupby('ptid').rid.nunique()>1).sum())
        item['missing_ptid_rows']=safe_count((~present(df['PTID'])).sum())
    KEY_REPORT[role]=item
if not all(KEY_REPORT[r]['missing_rid_rows']==0 for r in FILES):
    raise ValueError('STOP: missing participant RID found; clean source identifiers before joining.')
OVERLAP={
  'longitudinal_and_landmark12':safe_count(len(IDS['longitudinal']&IDS['landmark12'])),
  'landmark_not_in_longitudinal':safe_count(len(IDS['landmark12']-IDS['longitudinal'])),
  'longitudinal_not_in_landmark':safe_count(len(IDS['longitudinal']-IDS['landmark12'])),
  'repeat_pairs_participants_in_longitudinal':safe_count(len(IDS['repeat_pairs']&IDS['longitudinal'])),
  'repeat_pairs_participants_in_landmark':safe_count(len(IDS['repeat_pairs']&IDS['landmark12'])),
}
print(json.dumps({'identifier_integrity':KEY_REPORT,'participant_overlap':OVERLAP},indent=2))

{
  "identifier_integrity": {
    "longitudinal": {
      "rows": 258,
      "valid_rid_rows": 258,
      "missing_rid_rows": 0,
      "distinct_rid": 258,
      "duplicate_rid_rows": 0,
      "rid_whitespace_rows": 0,
      "rid_with_multiple_ptid": 0,
      "ptid_with_multiple_rid": 0,
      "missing_ptid_rows": 0
    },
    "repeat_pairs": {
      "rows": 90,
      "valid_rid_rows": 90,
      "missing_rid_rows": 0,
      "distinct_rid": 70,
      "duplicate_rid_rows": 36,
      "rid_whitespace_rows": 0
    },
    "landmark12": {
      "rows": 199,
      "valid_rid_rows": 199,
      "missing_rid_rows": 0,
      "distinct_rid": 199,
      "duplicate_rid_rows": 0,
      "rid_whitespace_rows": 0,
      "rid_with_multiple_ptid": 0,
      "ptid_with_multiple_rid": 0,
      "missing_ptid_rows": 0
    }
  },
  "participant_overlap": {
    "longitudinal_and_landmark12": 199,
    "landmark_not_in_longitudinal": 0,
    "longitudinal_not_in_landmark": 59,
    "repeat_pairs_participants_in_longi

In [6]:
LONG=DATA['longitudinal']
LAND=DATA['landmark12']
REPEAT=DATA['repeat_pairs']
ID_COLS=['BL_IMAGEUID','BL_LONIUID','M12_IMAGEUID','M12_LONIUID']
DATES=['BL_EXAMDATE','M12_EXAMDATE']
DATE_REPORT={}
IMAGE_REPORT={}
for role,df in [('longitudinal',LONG),('landmark12',LAND)]:
    parse={c:pd.to_datetime(df[c].where(present(df[c])),errors='coerce') for c in DATES}
    comparison=parse['BL_EXAMDATE'].notna() & parse['M12_EXAMDATE'].notna()
    delta=(parse['M12_EXAMDATE']-parse['BL_EXAMDATE']).dt.days
    DATE_REPORT[role]={
      'date_coverage':count_missing(df,DATES),
      'invalid_nonempty_dates':{c:safe_count((present(df[c])&parse[c].isna()).sum()) for c in DATES},
      'valid_date_pairs':safe_count(comparison.sum()),
      'm12_date_not_after_baseline':safe_count((comparison&(delta<=0)).sum()),
      'interval_days_median_on_valid_pairs':float(delta[comparison].median()) if comparison.any() else None,
    }
    ids={}
    for col in ID_COLS:
        if col not in df: continue
        valid=present(df[col]); s=clean(df[col])[valid]
        ids[col]={'nonmissing_rows':safe_count(valid.sum()),'distinct_values':safe_count(s.nunique()),
                  'shared_by_multiple_rid':safe_count(
                    (pd.DataFrame({'id':clean(df[col])[valid],'rid':clean(df['RID'])[valid]})
                        .groupby('id').rid.nunique()>1).sum())}
    images_equal = present(df['BL_IMAGEUID']) & present(df['M12_IMAGEUID']) & (
        clean(df['BL_IMAGEUID'])==clean(df['M12_IMAGEUID']))
    ids['baseline_m12_same_imageuid_rows']=safe_count(images_equal.sum())
    IMAGE_REPORT[role]=ids
print(json.dumps({'dates':DATE_REPORT,'source_identifier_integrity':IMAGE_REPORT},indent=2))

{
  "dates": {
    "longitudinal": {
      "date_coverage": {
        "BL_EXAMDATE": {
          "present": 258,
          "missing": 0
        },
        "M12_EXAMDATE": {
          "present": 258,
          "missing": 0
        }
      },
      "invalid_nonempty_dates": {
        "BL_EXAMDATE": 0,
        "M12_EXAMDATE": 0
      },
      "valid_date_pairs": 258,
      "m12_date_not_after_baseline": 0,
      "interval_days_median_on_valid_pairs": 387.0
    },
    "landmark12": {
      "date_coverage": {
        "BL_EXAMDATE": {
          "present": 199,
          "missing": 0
        },
        "M12_EXAMDATE": {
          "present": 199,
          "missing": 0
        }
      },
      "invalid_nonempty_dates": {
        "BL_EXAMDATE": 0,
        "M12_EXAMDATE": 0
      },
      "valid_date_pairs": 199,
      "m12_date_not_after_baseline": 0,
      "interval_days_median_on_valid_pairs": 387.0
    }
  },
  "source_identifier_integrity": {
    "longitudinal": {
      "BL_IMAGEUID": {
   

In [7]:
JOIN_REPORT={'status':'NOT_EVALUATED'}
if clean(LONG['RID']).is_unique and clean(LAND['RID']).is_unique:
    longitudinal_by_rid=LONG.set_index(clean(LONG['RID']))
    landmark_by_rid=LAND.set_index(clean(LAND['RID']))
    shared_rids=longitudinal_by_rid.index.intersection(landmark_by_rid.index)
    common=[col for col in LONG.columns if col in LAND.columns and col not in {'RID','PTID'}]
    mismatches={}
    any_mismatch=pd.Series(False,index=shared_rids)
    for col in common:
        a=clean(longitudinal_by_rid.loc[shared_rids,col])
        b=clean(landmark_by_rid.loc[shared_rids,col])
        diff=(a!=b)
        mismatches[col]=safe_count(diff.sum())
        any_mismatch |= diff
    JOIN_REPORT={
       'status':'CHECKED_USING_EXACT_RID',
       'compared_shared_columns':len(common),
       'joined_participants':safe_count(len(shared_rids)),
       'participants_with_any_shared_column_disagreement':safe_count(any_mismatch.sum()),
       'per_column_disagreements':mismatches,
    }
else:
    JOIN_REPORT={'status':'BLOCKED_DUPLICATE_RID','note':'No arbitrary deduplication or row selection performed.'}
print(json.dumps(JOIN_REPORT,indent=2))

{
  "status": "CHECKED_USING_EXACT_RID",
  "compared_shared_columns": 49,
  "joined_participants": 199,
  "participants_with_any_shared_column_disagreement": 0,
  "per_column_disagreements": {
    "BL_VISCODE": 0,
    "BL_EXAMDATE": 0,
    "BL_IMAGEUID": 0,
    "BL_LONIUID": 0,
    "BL_VERSION": 0,
    "BL_FLDSTRENG": 0,
    "BL_STATUS": 0,
    "BL_OVERALLQC": 0,
    "BL_TEMPQC": 0,
    "BL_VENTQC": 0,
    "BL_LHIPPOC": 0,
    "BL_RHIPPOC": 0,
    "BL_LENTORHIN": 0,
    "BL_RENTORHIN": 0,
    "BL_LMIDTEMP": 0,
    "BL_RMIDTEMP": 0,
    "BL_VENTRICLES": 0,
    "BL_ICV": 0,
    "M12_VISCODE": 0,
    "M12_EXAMDATE": 0,
    "M12_IMAGEUID": 0,
    "M12_LONIUID": 0,
    "M12_VERSION": 0,
    "M12_FLDSTRENG": 0,
    "M12_STATUS": 0,
    "M12_OVERALLQC": 0,
    "M12_TEMPQC": 0,
    "M12_VENTQC": 0,
    "M12_LHIPPOC": 0,
    "M12_RHIPPOC": 0,
    "M12_LENTORHIN": 0,
    "M12_RENTORHIN": 0,
    "M12_LMIDTEMP": 0,
    "M12_RMIDTEMP": 0,
    "M12_VENTRICLES": 0,
    "M12_ICV": 0,
    "BASELINE_DAT

In [8]:
uid_index={}
for col in ID_COLS:
    uid_index[col]={}
    for rid,uid in zip(clean(LONG['RID']),clean(LONG[col])):
        if uid and uid.lower() not in {'na','nan','null','none','unknown'}:
            uid_index[col].setdefault(rid,set()).add(uid)

repeat_evidence=Counter()
seen_pairs=set()
for rec in REPEAT.itertuples(index=False):
    row=dict(zip(REPEAT.columns,rec))
    rid=str(row['RID']).strip()
    a=str(row['IMAGEUID_1']).strip(); b=str(row['IMAGEUID_2']).strip()
    la=str(row['LONIUID_1']).strip(); lb=str(row['LONIUID_2']).strip()
    if a and b and a==b: repeat_evidence['same_imageuid_pair_rows']+=1
    if la and lb and la==lb: repeat_evidence['same_loniuid_pair_rows']+=1
    if a and b:
        signature=(rid,tuple(sorted((a,b))))
        if signature in seen_pairs: repeat_evidence['repeated_unordered_imageuid_pair_rows']+=1
        seen_pairs.add(signature)
    if any(u in uid_index['BL_IMAGEUID'].get(rid,set()) for u in (a,b) if u):
        repeat_evidence['rows_with_baseline_imageuid_link']+=1
    if any(u in uid_index['M12_IMAGEUID'].get(rid,set()) for u in (a,b) if u):
        repeat_evidence['rows_with_m12_imageuid_link']+=1
    if any(u in uid_index['BL_LONIUID'].get(rid,set()) for u in (la,lb) if u):
        repeat_evidence['rows_with_baseline_loniuid_link']+=1
    if any(u in uid_index['M12_LONIUID'].get(rid,set()) for u in (la,lb) if u):
        repeat_evidence['rows_with_m12_loniuid_link']+=1
    if not a or not b: repeat_evidence['rows_with_missing_imageuid_member']+=1
repeat_evidence['source_repeat_rows']=len(REPEAT)
repeat_evidence['unique_unordered_imageuid_pairs_with_both_members']=len(seen_pairs)
REPEAT_REPORT={key:safe_count(value) for key,value in sorted(repeat_evidence.items())}
REPEAT_REPORT['same_day_two_acquisitions_independently_verified']=False
REPEAT_REPORT['note']='A single EXAMDATE per repeat-pair row cannot independently verify both acquisitions\' dates.'
print(json.dumps(REPEAT_REPORT,indent=2))

{
  "same_loniuid_pair_rows": 16,
  "source_repeat_rows": 90,
  "unique_unordered_imageuid_pairs_with_both_members": 90,
  "same_day_two_acquisitions_independently_verified": false,
  "note": "A single EXAMDATE per repeat-pair row cannot independently verify both acquisitions' dates."
}


In [9]:
FEATURES={
 'LHIPPOC':('hippocampus','left'), 'RHIPPOC':('hippocampus','right'),
 'LENTORHIN':('entorhinal','left'), 'RENTORHIN':('entorhinal','right'),
 'LMIDTEMP':('middle_temporal','left'), 'RMIDTEMP':('middle_temporal','right'),
 'VENTRICLES':('ventricles','unspecified'), 'ICV':('intracranial_volume','not_applicable'),
}
EVENTS=[]
BAD_VALUES=Counter()
for rec in LONG.itertuples(index=False,name=None):
    row=dict(zip(LONG.columns,rec))
    for tp in ('BL','M12'):
        for source_feature,(region,laterality) in FEATURES.items():
            col=f'{tp}_{source_feature}'
            raw=str(row[col]).strip()
            if raw.lower() in {'','na','n/a','nan','none','null','unknown'}:
                continue
            try:
                value=float(raw)
            except (ValueError,TypeError):
                BAD_VALUES[(tp,source_feature)]+=1; continue
            if not math.isfinite(value):
                BAD_VALUES[(tp,source_feature)]+=1; continue
            EVENTS.append({'rid':str(row['RID']).strip(),'timepoint':tp,
                 'source_feature':source_feature,'anatomical_region':region,
                 'laterality':laterality,'value_raw':raw,'value_numeric':value,
                 'date_raw':str(row[f'{tp}_EXAMDATE']).strip(),
                 'source_imageuid_raw':str(row[f'{tp}_IMAGEUID']).strip(),
                 'source_loniuid_raw':str(row[f'{tp}_LONIUID']).strip(),
                 'source_version_raw':str(row.get(f'{tp}_VERSION','')).strip(),
                 'qc_overall_raw':str(row.get(f'{tp}_OVERALLQC','')).strip(),
                 'qc_temp_raw':str(row.get(f'{tp}_TEMPQC','')).strip(),
                 'qc_vent_raw':str(row.get(f'{tp}_VENTQC','')).strip(),
                 'unit_verified':False,'trace_state_verified':False})
EV=pd.DataFrame(EVENTS)
EV['date_valid']=pd.to_datetime(EV['date_raw'].where(EV['date_raw'].ne('')),errors='coerce').notna()
EV['source_image_id_present']=EV['source_imageuid_raw'].ne('') | EV['source_loniuid_raw'].ne('')
feature_counts=EV.groupby(['timepoint','source_feature']).size().to_dict()
EVIDENCE_REPORT={
  'status':'REAL_WIDE_SOURCE_TO_NORMALIZED_EVENTS_IN_MEMORY',
  'source_longitudinal_rows':int(len(LONG)),
  'generated_valid_numeric_event_rows':int(len(EV)),
  'events_with_valid_date':safe_count(EV.date_valid.sum()),
  'events_with_at_least_one_raw_image_identifier':safe_count(EV.source_image_id_present.sum()),
  'invalid_nonblank_numeric_values':{f'{tp}_{ft}':safe_count(n) for (tp,ft),n in BAD_VALUES.items()},
  'event_count_by_source_column':{f'{tp}_{ft}':safe_count(n) for (tp,ft),n in feature_counts.items()},
  'source_measurement_unit':'UNVERIFIED_IN_PROVIDED_CSV',
  'model_name':'NOT_VERIFIED_FROM_VERSION_COLUMN',
  'dicom_study_uid':'NOT_VERIFIED_FROM_IMAGEUID_OR_LONIUID',
  'original_trace_state':'NOT_PRESENT_IN_THREE_SOURCE_CSV_FILES',
}
print(json.dumps(EVIDENCE_REPORT,indent=2))

{
  "status": "REAL_WIDE_SOURCE_TO_NORMALIZED_EVENTS_IN_MEMORY",
  "source_longitudinal_rows": 258,
  "generated_valid_numeric_event_rows": 4128,
  "events_with_valid_date": 4128,
  "events_with_at_least_one_raw_image_identifier": 4128,
  "invalid_nonblank_numeric_values": {},
  "event_count_by_source_column": {
    "BL_ICV": 258,
    "BL_LENTORHIN": 258,
    "BL_LHIPPOC": 258,
    "BL_LMIDTEMP": 258,
    "BL_RENTORHIN": 258,
    "BL_RHIPPOC": 258,
    "BL_RMIDTEMP": 258,
    "BL_VENTRICLES": 258,
    "M12_ICV": 258,
    "M12_LENTORHIN": 258,
    "M12_LHIPPOC": 258,
    "M12_LMIDTEMP": 258,
    "M12_RENTORHIN": 258,
    "M12_RHIPPOC": 258,
    "M12_RMIDTEMP": 258,
    "M12_VENTRICLES": 258
  },
  "source_measurement_unit": "UNVERIFIED_IN_PROVIDED_CSV",
  "model_name": "NOT_VERIFIED_FROM_VERSION_COLUMN",
  "dicom_study_uid": "NOT_VERIFIED_FROM_IMAGEUID_OR_LONIUID",
  "original_trace_state": "NOT_PRESENT_IN_THREE_SOURCE_CSV_FILES"
}


In [10]:
source_bilateral=(
    present(LONG['BL_LHIPPOC']) & present(LONG['BL_RHIPPOC']) &
    present(LONG['M12_LHIPPOC']) & present(LONG['M12_RHIPPOC'])
)
num_bilateral=pd.Series(True,index=LONG.index)
for col in ('BL_LHIPPOC','BL_RHIPPOC','M12_LHIPPOC','M12_RHIPPOC'):
    numeric=pd.to_numeric(LONG[col].where(present(LONG[col])),errors='coerce')
    num_bilateral &= numeric.notna() & numeric.map(lambda v: bool(pd.notna(v) and math.isfinite(v) and v>0))
bl_date=pd.to_datetime(LONG['BL_EXAMDATE'].where(present(LONG['BL_EXAMDATE'])),errors='coerce')
m12_date=pd.to_datetime(LONG['M12_EXAMDATE'].where(present(LONG['M12_EXAMDATE'])),errors='coerce')
valid_time=bl_date.notna() & m12_date.notna() & (m12_date>bl_date)
has_both_ids=(present(LONG['BL_IMAGEUID'])|present(LONG['BL_LONIUID'])) & (
               present(LONG['M12_IMAGEUID'])|present(LONG['M12_LONIUID']))
CANDIDATE_REPORT={
 'not_a_prespecified_mci_cohort':True,
 'reason':'Baseline MCI diagnosis, measurement units and DICOM study identity have not been verified.',
 'nonmissing_bilateral_hippocampal_values':safe_count(source_bilateral.sum()),
 'positive_numeric_bilateral_hippocampal_values':safe_count(num_bilateral.sum()),
 'valid_ordered_baseline_m12_dates':safe_count(valid_time.sum()),
 'at_least_one_raw_image_id_each_timepoint':safe_count(has_both_ids.sum()),
 'all_four_conditions':safe_count((source_bilateral&num_bilateral&valid_time&has_both_ids).sum()),
}
print(json.dumps(CANDIDATE_REPORT,indent=2))

{
  "not_a_prespecified_mci_cohort": true,
  "reason": "Baseline MCI diagnosis, measurement units and DICOM study identity have not been verified.",
  "nonmissing_bilateral_hippocampal_values": 258,
  "positive_numeric_bilateral_hippocampal_values": 258,
  "valid_ordered_baseline_m12_dates": 258,
  "at_least_one_raw_image_id_each_timepoint": 258,
  "all_four_conditions": 258
}


In [11]:
UPSTREAM={
 'ohdsi_tfl':{'url':'https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git',
               'commit':'a4b7265adabf096e84b957de068e24a5ab853425'},
 'imagewg':{'url':'https://github.com/OHDSI/ImageWG.git',
            'commit':'8f0a19bb94e941bb335a097bc9851fab672621c7'},
 'pyomop':{'url':'https://github.com/dermatologist/pyomop.git',
           'commit':'dfaa6d4062dbc600f52f5377019a347c4caa2cc7'},
}
UPSTREAM_STATUS={}
for name, spec in UPSTREAM.items():
    dest=RUN/'verified_upstream'/name
    try:
        if not (dest/'.git').is_dir():
            dest.mkdir(parents=True,exist_ok=True)
            subprocess.run(['git','init','-q',str(dest)],check=True,timeout=30,capture_output=True,text=True)
            subprocess.run(['git','-C',str(dest),'remote','add','origin',spec['url']],check=True,timeout=30,capture_output=True,text=True)
        subprocess.run(['git','-C',str(dest),'fetch','-q','--depth','1','origin',spec['commit']],
                       check=True,timeout=120,capture_output=True,text=True)
        subprocess.run(['git','-C',str(dest),'checkout','-q','--detach','FETCH_HEAD'],
                       check=True,timeout=40,capture_output=True,text=True)
        actual=subprocess.run(['git','-C',str(dest),'rev-parse','HEAD'],
                              check=True,timeout=15,capture_output=True,text=True).stdout.strip()
        UPSTREAM_STATUS[name]={'status':'PINNED_AND_VERIFIED' if actual==spec['commit'] else 'COMMIT_MISMATCH',
                               'expected_commit':spec['commit'],'actual_commit':actual}
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        UPSTREAM_STATUS[name]={'status':'NOT_VERIFIED_THIS_RUN','expected_commit':spec['commit'],
                               'error_type':type(exc).__name__}
print(json.dumps(UPSTREAM_STATUS,indent=2))

{
  "ohdsi_tfl": {
    "status": "PINNED_AND_VERIFIED",
    "expected_commit": "a4b7265adabf096e84b957de068e24a5ab853425",
    "actual_commit": "a4b7265adabf096e84b957de068e24a5ab853425"
  },
  "imagewg": {
    "status": "PINNED_AND_VERIFIED",
    "expected_commit": "8f0a19bb94e941bb335a097bc9851fab672621c7",
    "actual_commit": "8f0a19bb94e941bb335a097bc9851fab672621c7"
  },
  "pyomop": {
    "status": "PINNED_AND_VERIFIED",
    "expected_commit": "dfaa6d4062dbc600f52f5377019a347c4caa2cc7",
    "actual_commit": "dfaa6d4062dbc600f52f5377019a347c4caa2cc7"
  }
}


In [12]:
SCHEMA_REPORT={'status':'NO_PINNED_SCHEMA_FIELDS_VERIFIED_IN_THIS_RUN'}
wg=RUN/'verified_upstream'/'imagewg'
field_csv=wg/'inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv'
if UPSTREAM_STATUS['imagewg']['status']=='PINNED_AND_VERIFIED' and field_csv.is_file():
    with field_csv.open(encoding='utf-8-sig',newline='') as f:
        reader=csv.DictReader(f)
        headers=list(reader.fieldnames or [])
        schema_rows=list(reader)
    # The field-level CSV is evidence of a draft extension; it is not a verified executable DDL.
    rows_with_image_occurrence=sum('IMAGE_OCCURRENCE' in ' '.join(str(v).upper() for v in r.values()) for r in schema_rows)
    rows_with_image_feature=sum('IMAGE_FEATURE' in ' '.join(str(v).upper() for v in r.values()) for r in schema_rows)
    SCHEMA_REPORT={'status':'PINNED_FIELD_LEVEL_CSV_READ_NOT_EXECUTABLE_DDL',
       'csv_sha256':sha256_file(field_csv),'header_columns':headers,
       'source_field_spec_rows':len(schema_rows),
       'rows_mentioning_image_occurrence':rows_with_image_occurrence,
       'rows_mentioning_image_feature':rows_with_image_feature,
       'ddl_executed':False}
TFL_REPORT={'status':'NOT_INSPECTED_THIS_RUN'}
tfl=RUN/'verified_upstream'/'ohdsi_tfl'
if UPSTREAM_STATUS['ohdsi_tfl']['status']=='PINNED_AND_VERIFIED':
    symbols=re.compile(r'PRESERVED|NORMALIZED|COLLAPSED|AMBIGUOUS|TRACEABILITY_LOSS|TFL',re.I)
    found=[]
    for path in tfl.rglob('*.py'):
        if '.git' in path.parts: continue
        body=path.read_text(encoding='utf-8',errors='replace')
        if symbols.search(body):
            found.append({'source_path':str(path.relative_to(tfl)),
                          'sha256':sha256_file(path),
                          'occurrences':len(symbols.findall(body))})
    TFL_REPORT={'status':'STATIC_CANDIDATES_ONLY_NOT_EXECUTED',
                'matching_python_files':found,'frozen_original_adapter_executed':False}
print(json.dumps({'imagewg_field_spec':SCHEMA_REPORT,'original_tfl_source_scan':TFL_REPORT},indent=2))

{
  "imagewg_field_spec": {
    "status": "PINNED_FIELD_LEVEL_CSV_READ_NOT_EXECUTABLE_DDL",
    "csv_sha256": "af2e1d26aae56d05f3cc5d90f5ac3d523b64a15dcb4e8ee038a5c2a0aa129cf4",
    "header_columns": [
      "cdmTableName",
      "cdmFieldName",
      "isRequired\u00a0",
      "cdmDatatype\u00a0",
      "useGuidance",
      "etlConventions",
      "isPrimaryKey",
      "isForeignKey",
      "fkTableName",
      "fkFrieldName",
      "fkDomain",
      "fkClass"
    ],
    "source_field_spec_rows": 23,
    "rows_mentioning_image_occurrence": 13,
    "rows_mentioning_image_feature": 12,
    "ddl_executed": false
  },
  "original_tfl_source_scan": {
    "status": "STATIC_CANDIDATES_ONLY_NOT_EXECUTED",
    "matching_python_files": [],
    "frozen_original_adapter_executed": false
  }
}


In [13]:
TRACE_COLS={role:[col for col in df.columns if re.search(
    r'(^|_)(TRACE|CAUTION|REVIEW_REQUIRED|CALIBRATION|RELIABILITY)(_|$)',col,re.I)]
            for role,df in DATA.items()}
UNIT_COLS={role:[col for col in df.columns if re.search(r'(^|_)(UNIT|UCUM|MM3|CM3)(_|$)',col,re.I)]
           for role,df in DATA.items()}
BLOCKERS={
 'baseline_mci_status_documented':False,
 'measurement_unit_confirmed_from_authoritative_source':False,
 'imageuid_confirmed_to_be_dicom_study_uid':False,
 'series_or_acquisition_identifier_verified':False,
 'model_name_and_version_verified_as_software_identity':False,
 'original_trace_states_reference_and_rule_version_available':False,
 'original_tfl_v15_replayed_on_real_source':False,
 'omop_vocabulary_snapshot_pinned':False,
 'pinned_micdm_executable_ddl_loaded':False,
 'qualified_source_cohort_approved':False,
}
CONTRACT={
  'participant_key':'RID (joined exactly after stripping outer whitespace, no inferred substitute)',
  'alternate_key':'PTID (crosschecked but not substituted for RID)',
  'timepoints':['BL','M12'],
  'timepoint_fields':['BL_VISCODE','M12_VISCODE'],
  'image_id_fields':['BL_IMAGEUID','M12_IMAGEUID','BL_LONIUID','M12_LONIUID'],
  'source_date_fields':['BL_EXAMDATE','M12_EXAMDATE'],
  'source_numeric_columns':{'BL':[f'BL_{f}' for f in FEATURES],
                             'M12':[f'M12_{f}' for f in FEATURES]},
  'qc_as_uninterpreted_source_strings':['BL_OVERALLQC','M12_OVERALLQC',
                  'BL_TEMPQC','M12_TEMPQC','BL_VENTQC','M12_VENTQC'],
  'version_as_uninterpreted_source_strings':['BL_VERSION','M12_VERSION'],
  'diagnostic_LABEL_is_not_assumed_to_mean_baseline_mci':True,
  'unit':None,
  'dicom_study_uid':None,
  'trace_state':None,
  'trace_rule_version':None,
  'trace_calibration_reference':None,
  'allowed_to_load_omop':False,
  'allowed_to_report_etl_comparison':False,
}
REPORT={
  'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
  'phase':'03_verified_source_evidence',
  'run_utc':UTC_NOW,
  'scientific_results_status':'NO_ETL_COMPARISON_RUN',
  'actual_adni_source_files_present':True,
  'source_private_data_saved_to_drive':False,
  'environment':{'python':sys.version.split()[0], 'pandas':pd.__version__,
                 'platform':platform.platform()},
  'source_profiles':{r:{'rows':int(len(d)), 'columns':int(len(d.columns))} for r,d in DATA.items()},
  'source_file_hashes_intentionally_omitted_from_shareable_json':True,
  'source_identifier_integrity':KEY_REPORT,
  'source_cross_file_overlap':OVERLAP,
  'source_visit_dates':DATE_REPORT,
  'source_image_identifiers':IMAGE_REPORT,
  'wide_source_join':JOIN_REPORT,
  'repeat_pair_lineage':REPEAT_REPORT,
  'real_measurement_event_evidence':EVIDENCE_REPORT,
  'candidate_hippocampus_completeness_not_approved_cohort':CANDIDATE_REPORT,
  'evidence_contract':CONTRACT,
  'unverified_trace_column_candidates':TRACE_COLS,
  'unverified_unit_column_candidates':UNIT_COLS,
  'upstream_revisions':UPSTREAM_STATUS,
  'imagewg_specification':SCHEMA_REPORT,
  'frozen_generic_tfl_inventory':TFL_REPORT,
  'unresolved_requirements':BLOCKERS,
  'actual_fhir_r4_validated':False,
  'actual_micdm_target_loaded':False,
  'actual_comparator_scores':None,
  'next_stage_gate':'SOURCE_CROSSWALK_COMPLETED_REVIEW_PROVENANCE_AND_UNIT_BEFORE_FHIR_MICDM',
}
shareable=DEST/'ICHI2027_Phase03_SHAREABLE.json'
shareable.write_text(json.dumps(REPORT,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('Saved shareable audit:',shareable)
print(json.dumps({'phase':REPORT['phase'],'scientific_results_status':REPORT['scientific_results_status'],
                  'source_files':REPORT['source_profiles'],
                  'unresolved_requirements':BLOCKERS},indent=2))
if IS_COLAB and not DRIVE_OK:
    from google.colab import files as colab_files
    colab_files.download(str(shareable))

Saved shareable audit: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase03_outputs/ICHI2027_Phase03_SHAREABLE.json
{
  "phase": "03_verified_source_evidence",
  "scientific_results_status": "NO_ETL_COMPARISON_RUN",
  "source_files": {
    "longitudinal": {
      "rows": 258,
      "columns": 51
    },
    "repeat_pairs": {
      "rows": 90,
      "columns": 51
    },
    "landmark12": {
      "rows": 199,
      "columns": 70
    }
  },
  "unresolved_requirements": {
    "baseline_mci_status_documented": false,
    "measurement_unit_confirmed_from_authoritative_source": false,
    "imageuid_confirmed_to_be_dicom_study_uid": false,
    "series_or_acquisition_identifier_verified": false,
    "model_name_and_version_verified_as_software_identity": false,
    "original_trace_states_reference_and_rule_version_available": false,
    "original_tfl_v15_replayed_on_real_source": false,
    "omop_vocabulary_snapshot_pinned": false,
    "pinned_micdm_executable_ddl_loaded": false,
    "qualified_sou